# CASMI 2026 — streaming spectral retrieval baseline

This offline notebook runs the versioned source snapshot against the competition input and writes a validated `submission.csv`. The downloadable test parquet is a dummy; only grouped validation is meaningful for model selection.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

INPUT = Path('/kaggle/input')
competition_dirs = [p for p in INPUT.iterdir() if (p / 'train.parquet').exists() and (p / 'test.parquet').exists()]
source_files = list(INPUT.glob('**/src/retrieve.py'))
if not competition_dirs:
    raise FileNotFoundError('Attach the Enveda CASMI competition data input')
if not source_files:
    raise FileNotFoundError('Attach the versioned source-code Kaggle Dataset containing src/retrieve.py')
DATA_DIR = competition_dirs[0]
SOURCE_ROOT = source_files[0].parent.parent
sys.path.insert(0, str(SOURCE_ROOT / 'src'))
print('Data:', DATA_DIR)
print('Code snapshot:', SOURCE_ROOT)

# Use the exact RDKit release required by the scorer. Kaggle input should include its offline wheel.
try:
    import rdkit
    rdkit_version = rdkit.__version__
except ImportError:
    rdkit_version = None
if rdkit_version != '2026.03.3':
    wheels = list(INPUT.glob('**/*rdkit*.whl'))
    if not wheels:
        raise RuntimeError('Attach the offline RDKit 2026.03.3 wheel dataset (wheel filename should include rdkit)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', str(wheels[0])], check=True)
    import rdkit
    print('Installed RDKit', rdkit.__version__)

In [ ]:
from retrieve import retrieve
from write_submission import validate_predictions
import pandas as pd

predictions = retrieve(
    str(DATA_DIR / 'train.parquet'),
    str(DATA_DIR / 'test.parquet'),
    max_ppm=30.0,
    fallback_ppm=250.0,
    max_peaks=256,
    batch_size=8192,
    top_k=25,
]
test_ids = pd.read_parquet(DATA_DIR / 'test.parquet', columns=['molecule_id'])['molecule_id'].unique()
submission = validate_predictions(predictions, test_ids)
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.shape)
submission.head()

The local downloadable test is only a plumbing check. Kaggle replaces it with hidden spectra when a competition submission notebook is rerun. Keep this notebook offline and record the source Git SHA alongside each saved notebook version.